# ⚡ TalkForge — AI Talking-Head Video Generator

Upload a portrait image + audio → Get a lip-synced talking video.

**Steps:**
1. Make sure Runtime → Change runtime type → **T4 GPU** is selected
2. Click **Runtime → Run all**
3. Wait for all cells to complete (~5–10 minutes on first run)
4. Click the **public Gradio link** printed at the bottom

> No manual steps required. Everything installs and downloads automatically.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Cell 1 — Verify GPU
# ─────────────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    for line in result.stdout.split('\n')[:15]:
        print(line)
    print('\n✅ GPU detected!')
else:
    print('⚠️  No GPU detected.')
    print('   Go to Runtime → Change runtime type → Hardware accelerator → T4 GPU')
    print('   Then Runtime → Disconnect and delete runtime, then Run all again.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Cell 2 — Install system packages (FFmpeg, cmake, dlib deps)
# ─────────────────────────────────────────────────────────────────────
print('Installing system packages…')
!apt-get update -qq
!apt-get install -y -qq \
    ffmpeg \
    cmake \
    build-essential \
    libboost-all-dev \
    libgtk2.0-dev \
    pkg-config \
    libopenblas-dev \
    liblapack-dev \
    libatlas-base-dev \
    wget curl git unzip

# Confirm FFmpeg
!ffmpeg -version | head -1
print('✅ System packages ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Cell 3 — Clone TalkForge & SadTalker
# ─────────────────────────────────────────────────────────────────────
import os

TALKFORGE_DIR  = '/content/TalkForge'
SADTALKER_DIR  = '/content/SadTalker'

# Clone TalkForge
if not os.path.exists(TALKFORGE_DIR):
    print('Cloning TalkForge…')
    !git clone --depth 1 https://github.com/rydenxgod-bot/TalkForge.git {TALKFORGE_DIR}
else:
    print('TalkForge already cloned, pulling latest…')
    !cd {TALKFORGE_DIR} && git pull --ff-only

# Clone SadTalker (into TalkForge folder AND standalone)
SADTALKER_IN_TF = os.path.join(TALKFORGE_DIR, 'SadTalker')
if not os.path.exists(SADTALKER_IN_TF):
    print('Cloning SadTalker…')
    !git clone --depth 1 https://github.com/OpenTalker/SadTalker.git {SADTALKER_IN_TF}
else:
    print('SadTalker already cloned.')

print('✅ Repositories ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Cell 4 — Install Python dependencies
# ─────────────────────────────────────────────────────────────────────
import sys

SADTALKER_IN_TF = '/content/TalkForge/SadTalker'

print('Step 4a: Installing SadTalker requirements…')
!pip install -q -r {SADTALKER_IN_TF}/requirements.txt

print('Step 4b: Installing TalkForge requirements…')
!pip install -q \
    'gradio>=4.0.0' \
    'torch>=2.0.0' \
    torchvision \
    torchaudio \
    'numpy<2.0' \
    scipy \
    librosa \
    soundfile \
    pillow \
    opencv-python-headless \
    imageio \
    'imageio-ffmpeg' \
    tqdm \
    pyyaml \
    face_alignment \
    facexlib \
    gfpgan \
    'realesrgan' \
    basicsr \
    safetensors \
    batch-face \
    kornia \
    einops \
    yacs \
    mediapipe \
    dlib

print('✅ Python dependencies installed.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Cell 5 — Download SadTalker model weights
# ─────────────────────────────────────────────────────────────────────
import os

WEIGHTS_DIR     = '/content/TalkForge/weights'
SADTALKER_IN_TF = '/content/TalkForge/SadTalker'
CKPT_DIR        = os.path.join(SADTALKER_IN_TF, 'checkpoints')

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

HF = 'https://huggingface.co/vinthony/SadTalker/resolve/main'

FILES = [
    ('SadTalker_V0.0.2_256.safetensors', f'{HF}/SadTalker_V0.0.2_256.safetensors'),
    ('SadTalker_V0.0.2_512.safetensors', f'{HF}/SadTalker_V0.0.2_512.safetensors'),
    ('mapping_00109-model.pth.tar',       f'{HF}/mapping_00109-model.pth.tar'),
    ('mapping_00229-model.pth.tar',       f'{HF}/mapping_00229-model.pth.tar'),
    ('epoch_20.pth',                      f'{HF}/epoch_20.pth'),
]

for fname, url in FILES:
    dest = os.path.join(WEIGHTS_DIR, fname)
    ckpt = os.path.join(CKPT_DIR, fname)
    if not os.path.exists(dest):
        print(f'Downloading {fname}…')
        !wget -q --show-progress -O {dest} "{url}"
    else:
        print(f'✓ {fname} already present')
    # Symlink into SadTalker checkpoints dir
    if not os.path.exists(ckpt):
        os.symlink(dest, ckpt)

# BFM (Basel Face Model) data
BFM_ZIP  = os.path.join(WEIGHTS_DIR, 'BFM_Fitting.zip')
BFM_DIR  = os.path.join(WEIGHTS_DIR, 'BFM_Fitting')
BFM_CKPT = os.path.join(CKPT_DIR, 'BFM_Fitting')

if not os.path.exists(BFM_DIR):
    print('Downloading BFM_Fitting.zip…')
    !wget -q --show-progress -O {BFM_ZIP} "{HF}/BFM_Fitting.zip"
    !unzip -q {BFM_ZIP} -d {WEIGHTS_DIR}
    print('✓ BFM_Fitting extracted')
else:
    print('✓ BFM_Fitting already present')

if not os.path.exists(BFM_CKPT):
    os.symlink(BFM_DIR, BFM_CKPT)

print('✅ SadTalker weights ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Cell 6 — Download GFPGAN face enhancer weights
# ─────────────────────────────────────────────────────────────────────
import os

SADTALKER_IN_TF = '/content/TalkForge/SadTalker'
GFPGAN_DIR      = os.path.join(SADTALKER_IN_TF, 'gfpgan', 'weights')
os.makedirs(GFPGAN_DIR, exist_ok=True)

GFPGAN_WEIGHTS = [
    ('GFPGANv1.4.pth',
     'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'),
    ('detection_Resnet50_Final.pth',
     'https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth'),
    ('parsing_parsenet.pth',
     'https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth'),
]

for fname, url in GFPGAN_WEIGHTS:
    dest = os.path.join(GFPGAN_DIR, fname)
    if not os.path.exists(dest):
        print(f'Downloading {fname}…')
        !wget -q --show-progress -O {dest} "{url}"
    else:
        print(f'✓ {fname} already present')

# Also put weights in the default facexlib cache location
FACEXLIB_DIR = os.path.expanduser('~/.cache/facexlib')
os.makedirs(FACEXLIB_DIR, exist_ok=True)
for fname in ['detection_Resnet50_Final.pth', 'parsing_parsenet.pth']:
    src  = os.path.join(GFPGAN_DIR, fname)
    dest = os.path.join(FACEXLIB_DIR, fname)
    if os.path.exists(src) and not os.path.exists(dest):
        import shutil
        shutil.copy2(src, dest)

print('✅ GFPGAN / facexlib weights ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Cell 7 — Configure paths and run smoke test
# ─────────────────────────────────────────────────────────────────────
import sys, os

TALKFORGE_DIR   = '/content/TalkForge'
SADTALKER_IN_TF = os.path.join(TALKFORGE_DIR, 'SadTalker')

# Add to sys.path so all imports resolve
for p in [TALKFORGE_DIR, SADTALKER_IN_TF]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(TALKFORGE_DIR)

# Smoke tests
ok = True
tests = [
    ('torch',         'import torch; print("  CUDA:", torch.cuda.is_available(), "| Torch:", torch.__version__)'),
    ('gradio',        'import gradio; print("  Gradio:", gradio.__version__)'),
    ('cv2',           'import cv2; print("  OpenCV:", cv2.__version__)'),
    ('librosa',       'import librosa; print("  librosa:", librosa.__version__)'),
    ('face_alignment','import face_alignment; print("  face_alignment: OK")'),
    ('gfpgan',        'import gfpgan; print("  gfpgan: OK")'),
    ('safetensors',   'import safetensors; print("  safetensors: OK")'),
]

for name, stmt in tests:
    try:
        exec(stmt)
    except Exception as e:
        print(f'  ✗ {name}: {e}')
        ok = False

# Verify key checkpoint files
REQUIRED_FILES = [
    '/content/TalkForge/SadTalker/checkpoints/SadTalker_V0.0.2_256.safetensors',
    '/content/TalkForge/SadTalker/checkpoints/epoch_20.pth',
    '/content/TalkForge/SadTalker/gfpgan/weights/GFPGANv1.4.pth',
]
for f in REQUIRED_FILES:
    if os.path.exists(f):
        print(f'  ✓ {os.path.basename(f)}')
    else:
        print(f'  ✗ MISSING: {f}')
        ok = False

if ok:
    print('\n✅ All checks passed. Ready to launch!')
else:
    print('\n⚠️  Some checks failed. Re-run cells 5 and 6, then try again.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Cell 8 — Launch TalkForge 🚀
# ─────────────────────────────────────────────────────────────────────
import sys, os

TALKFORGE_DIR   = '/content/TalkForge'
SADTALKER_IN_TF = os.path.join(TALKFORGE_DIR, 'SadTalker')

for p in [TALKFORGE_DIR, SADTALKER_IN_TF]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(TALKFORGE_DIR)
os.makedirs(os.path.join(TALKFORGE_DIR, 'outputs'), exist_ok=True)

print('═' * 60)
print('  ⚡  TalkForge — Launching Gradio interface…')
print('  A public URL will appear below in ~30 seconds.')
print('═' * 60)

from app.ui import launch
launch(share=True, port=7860)